# Lesson 12 — CSV

Write your solutions in `solution.py` first, then paste into the code cells below.
Every cell is self-contained — it builds its own fixtures.

---

## E1 — Read and print [REAL-WORLD]

Read `students.csv` (provided in this lesson folder). Print each student's name and score, one per line.

Output:
```text
Alice: 92
Bob: 78
Carol: 88
Dan: 45
Eve: 95
```

Constraints:
- Use `csv.DictReader`.
- Use `newline=""` and `encoding="utf-8"`.
- Do not hardcode the data — read from the file.

In [ ]:
# E1 — your solution here
import csv

with open("students.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for line in reader:
        print(f"{line['name']}: {line['score']}")


Alice: 92
Bob: 78
Carol: 88
Dan: 45
Eve: 95


---

## E2 — Write a CSV [REAL-WORLD]

Write a CSV file `fruits.csv` with header `name,quantity,price` and three rows of your choice.

Rules:
- Use `csv.writer` and `writerow`/`writerows`.
- Header row must be written first.
- Output file must have exactly 4 lines (header + 3 data rows).

Constraints:
- Use `newline=""` and `encoding="utf-8"`.
- Use pathlib for the file path.

In [78]:
# E2 — your solution here
from pathlib import Path

fruit_path = Path("fruits.csv")

with fruit_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "quantity", "price"])
    writer.writerows([("apple", 5, 1500), ("banana", 2, 1200), ("watermelon", "4", "2500")])

with fruit_path.open(newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    count = sum(1 for _ in reader)
    print(count)

4


---

## E3 — Safe CSV loader [REAL-WORLD]

Write a function `load_scores(path)` that:
1. Reads a CSV file with `DictReader`.
2. Converts `score` to `int` for each row.
3. If the file does not exist, returns an empty list.
4. If a row has a non-numeric score, skips that row (prints a warning).
5. Returns a list of dicts: `[{"name": "Alice", "subject": "math", "score": 92}, ...]`.

Output (for a CSV with one bad row):
```text
skipped row: non-numeric score 'abc' (row 3)
```

Constraints:
- Use `newline=""` and `encoding="utf-8"`.
- Wrap `int()` in `try/except ValueError`.
- Follow the L9/L11 loader contract: return data, never error strings.
- Do not print the loaded data — return it.

In [67]:
# E3 — your solution here
def load_scores(path: Path):
    if not path.exists():
        return []
    
    valid = []
    
    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for i, row in enumerate(reader, start=1):
            try:
                row['score'] = int(row['score'])
                valid.append(row)
            except ValueError:
                print(f"skipped row: non-numeric score '{row['score']}' (row {i})")
    
    return valid

print(load_scores(Path("students.csv")))

skipped row: non-numeric score '92.4' (row 1)
skipped row: non-numeric score 'abc' (row 4)
[{'name': 'Bob', 'subject': 'science', 'score': 78}, {'name': 'Carol', 'subject': 'math', 'score': 88}, {'name': 'Eve', 'subject': 'math', 'score': 95}]


---

## Challenge — CSV Grade Book [REAL-WORLD]

Given a CSV file `grades.csv` with columns `name,subject,score`, write a program that:

1. Reads the CSV with `csv.DictReader`.
2. Converts `score` to `int` (skips rows with bad scores, prints a warning per skipped row).
3. Computes the **average score per student** (across all subjects).
4. Writes a summary CSV `summary.csv` with columns `name,average,status` where:
   - `average` is a float with 1 decimal place (e.g., `85.0`).
   - `status` is `"pass"` if average >= 60, `"fail"` otherwise.
5. Prints the summary to stdout in this exact format:

```text
Summary (3 students):
1. Alice: 88.3 [pass]
2. Bob: 72.0 [pass]
3. Carol: 55.0 [fail]
```

Rules:
- Use `csv.DictReader` for reading, `csv.DictWriter` for writing.
- Handle missing file -> print "no grades file found" and exit.
- Handle corrupt rows -> skip and warn, never crash.
- Output format is a contract — byte-exact.

Constraints:
- Use `newline=""` and `encoding="utf-8"`.
- Use pathlib for file paths.
- No classes. Functions only.
- Follow the L9/L11 loader contract.

Deliverable: `solution.py` (starts blank).

In [79]:
# Challenge — your solution here

def read_csv(path: Path):
    if not path.exists():
        raise ValueError("no grades file found")
    
    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        rows = [row for row in reader]
        return rows

def grades_stat_builder(rows):
    stats = {}
    stat_summaries = []
    
    for row in rows:
        name = row['name']
        
        if name not in stats:
            stats[name] = {
                "total_score": 0,
                "count": 0,
            }
        
        stats[name]['total_score'] += row['score']
        stats[name]['count'] += 1
        
    for name in sorted(stats):
        if stats[name]['count'] > 0:
            average = stats[name]['total_score']/stats[name]['count']
            status = 'pass' if average >= 60 else 'fail'
            stat_summaries.append({"name": name, "average": f"{average:.1f}", "status": status})
        
    return stat_summaries

def write_csv(path: Path, data):
    field_names = list(data[0].keys())
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=field_names)
        writer.writeheader()
        writer.writerows(data)

def main():
    FILE_PATH = Path("grades.csv")
    
    rows = read_csv(FILE_PATH)
    parsed_rows = []
    
    for i, row in enumerate(rows, start=1):
        try:
            row['score'] = int(row['score'])
            parsed_rows.append(row)
        except ValueError:
            print(f"skipped row: non-numeric score '{row['score']}' (row {i})")
            continue
    
    stat_summaries = grades_stat_builder(parsed_rows)
    
    save_path = Path('summary.csv')
    write_csv(save_path,stat_summaries)
    
    results = read_csv(save_path)
    print(f"Summary ({len(results)} students):")
    for i, r in enumerate(results, start=1):
        print(f"{i}. {r['name']}: {r['average']} [{r['status']}]")

if __name__ == '__main__':
    main()

skipped row: non-numeric score 'abc' (row 13)
Summary (5 students):
1. Alice: 88.3 [pass]
2. Bob: 72.0 [pass]
3. Carol: 55.0 [fail]
4. Dan: 91.0 [pass]
5. Eve: 67.5 [pass]
